In [1]:
import lsdb
from pathlib import Path
import pyarrow.parquet as pq
import numpy as np
import matplotlib.pyplot as plt
from lsst.daf.butler import Butler

INFO:numexpr.utils:Note: detected 128 virtual cores but NumExpr set to maximum of 64, check "NUMEXPR_MAX_THREADS" environment variable.
INFO:numexpr.utils:Note: NumExpr detected 128 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO:numexpr.utils:NumExpr defaulting to 16 threads.


In [2]:
repo = 'embargo'
collection = "LSSTCam/prompt/output-20260127"

butler = Butler(repo, collections=collection)

INFO:botocore.credentials:Found credentials in shared credentials file: /sdf/home/n/ncaplar/.lsst/aws-credentials.ini


In [3]:
# List all data product names available
available_products = set()
for dataset_type in butler.registry.queryDatasetTypes():
    available_products.add(dataset_type.name)

# they are not in prompt outputs 
"""
print("\nAvailable data products:")
for product in sorted(available_products):
    print(f"  {product}")
"""

'\nprint("\nAvailable data products:")\nfor product in sorted(available_products):\n    print(f"  {product}")\n'

In [6]:
repo = 'embargo'
collection = "LSSTCam/runs/nightlyValidation"

butler = Butler(repo, collections=collection)

In [5]:
# List all data product names available
available_products = set()
for dataset_type in butler.registry.queryDatasetTypes():
    available_products.add(dataset_type.name)

"""
print("\nAvailable data products:")
for product in sorted(available_products):
    print(f"  {product}")
"""

'\nprint("\nAvailable data products:")\nfor product in sorted(available_products):\n    print(f"  {product}")\n'

In [6]:
# Get the first dataset reference
dataset_refs = list(butler.registry.queryDatasets(
    'single_visit_star_footprints',
    collections=collection
))

print(f"Found {len(dataset_refs)} single_visit_star_footprints")

# Load the first one
if dataset_refs:
    first_footprint = butler.get(dataset_refs[0])
    print(f"Loaded: {dataset_refs[0].dataId}")
    print(f"Type: {type(first_footprint)}")

Found 1788321 single_visit_star_footprints
Loaded: {instrument: 'LSSTCam', detector: 185, visit: 2025121900017, band: 'r', day_obs: 20251219, physical_filter: 'r_57'}
Type: <class 'lsst.afw.table.SourceCatalog'>


In [7]:
refs = list(
    butler.registry.queryDatasets(
        "single_visit_star_footprints",
        collections=collection,
        where="instrument='LSSTCam' AND exposure.day_obs = day",
        bind={"day": 20250703},
    )
)

In [8]:
refs

[]

In [7]:
# ...existing code...

import random
import pandas as pd
from tqdm.auto import tqdm

# Get all refs (needed to randomly sample)
all_refs = list(
    butler.registry.queryDatasets(
        "single_visit_star_footprints",
        collections=collection,
    )
)
print(f"Found {len(all_refs)} single_visit_star_footprints")

# Sample up to 100 at random
random.seed(48)
n = min(100, len(all_refs))
sample_refs = random.sample(all_refs, n)

dfs = []
for ref in tqdm(sample_refs, total=len(sample_refs), desc="Loading + converting single_visit_star_footprints"):
    cat = butler.get(ref)  # lsst.afw.table.SourceCatalog
    df = cat.asAstropy().to_pandas()

    # Keep provenance safely (DataCoordinate -> plain dict)
    data_id = None

    if hasattr(ref.dataId, "toDict"):
        data_id = ref.dataId.toDict()
    elif hasattr(ref.dataId, "to_simple"):
        simple = ref.dataId.to_simple()
        if hasattr(simple, "model_dump"):          # pydantic v2
            data_id = simple.model_dump()
        elif hasattr(simple, "dict"):              # pydantic v1 fallback
            data_id = simple.dict()
        else:
            data_id = None

    if not isinstance(data_id, dict):
        data_id = {"dataId": str(ref.dataId)}

    for k, v in data_id.items():
        df[k] = v

    dfs.append(df)

all_sources_df = pd.concat(dfs, ignore_index=True, sort=False)
print(all_sources_df.shape)
all_sources_df.head()


Found 1788321 single_visit_star_footprints


Loading + converting single_visit_star_footprints:   0%|          | 0/100 [00:00<?, ?it/s]

(140837, 500)


,id,coord_ra,coord_dec,parent,calib_psf_candidate,calib_psf_used,calib_psf_reserved,calib_astrometry_used,apcorr_slot_CalibFlux_used,apcorr_base_GaussianFlux_used,...,base_PsfFlux_mag,slot_PsfFlux_mag,base_PsfFlux_fluxErr,slot_PsfFlux_fluxErr,base_PsfFlux_magErr,slot_PsfFlux_magErr,deblend_psf_flux,deblend_psf_mag,dataId,records
0,25741821864837131,2.264783,0.260910,0,False,False,False,False,False,False,...,21.730606,21.730606,511.013471,511.013471,0.075231,0.075231,NaN,NaN,NaN,None
1,25741821864837135,2.264707,0.260815,0,False,False,False,False,False,False,...,19.846755,19.846755,544.322290,544.322290,0.014135,0.014135,NaN,NaN,NaN,None
2,25741821864837137,2.267119,0.262996,0,True,True,False,True,False,True,...,18.091569,18.091569,706.558427,706.558427,0.003643,0.003643,NaN,NaN,NaN,None
3,25741821864837140,2.267027,0.262899,0,True,True,False,False,False,True,...,16.444577,16.444577,1191.702103,1191.702103,0.001348,0.001348,NaN,NaN,NaN,None
4,25741821864837144,2.267292,0.263116,0,False,False,False,False,False,False,...,19.059714,19.059714,584.216254,584.216254,0.007348,0.007348,NaN,NaN,NaN,None


In [8]:
all_sources_df.tail(20)

,id,coord_ra,coord_dec,parent,calib_psf_candidate,calib_psf_used,calib_psf_reserved,calib_astrometry_used,apcorr_slot_CalibFlux_used,apcorr_base_GaussianFlux_used,...,base_PsfFlux_mag,slot_PsfFlux_mag,base_PsfFlux_fluxErr,slot_PsfFlux_fluxErr,base_PsfFlux_magErr,slot_PsfFlux_magErr,deblend_psf_flux,deblend_psf_mag,dataId,records
140817,25772622312114929,2.389098,-0.056533,25772622312114181,False,False,False,True,False,False,...,17.216683,17.216683,773.816832,773.816832,0.001783,0.001783,NaN,NaN,NaN,None
140818,25772622312114933,2.388764,-0.054533,25772622312114205,False,False,False,True,False,False,...,17.618244,17.618244,697.408618,697.408618,0.002325,0.002325,NaN,NaN,NaN,None
140819,25772622312114935,2.389284,-0.056277,25772622312114312,False,False,False,False,False,False,...,21.253553,21.253553,275.055937,275.055937,0.026096,0.026096,NaN,NaN,NaN,None
140820,25772622312114936,2.389260,-0.056263,25772622312114312,False,False,False,False,False,False,...,23.318352,23.318352,257.331393,257.331393,0.163515,0.163515,1309.123886,23.607548,NaN,None
140821,25772622312114938,2.389194,-0.055439,25772622312114330,False,False,False,False,False,False,...,20.418786,20.418786,319.156400,319.156400,0.014036,0.014036,22621.673233,20.513688,NaN,None
140822,25772622312114942,2.389164,-0.055342,25772622312114346,False,False,False,False,False,False,...,18.898333,18.898333,431.211234,431.211234,0.004675,0.004675,NaN,NaN,NaN,None
140823,25772622312114943,2.389184,-0.055312,25772622312114346,False,False,False,False,False,False,...,19.877501,19.877501,346.466165,346.466165,0.009255,0.009255,NaN,NaN,NaN,None
140824,25772622312114944,2.389193,-0.055353,25772622312114346,False,False,False,False,False,False,...,20.389558,20.389558,319.104567,319.104567,0.013661,0.013661,NaN,NaN,NaN,None
140825,25772622312114945,2.389325,-0.055488,25772622312114382,False,False,False,False,False,False,...,19.603088,19.603088,370.528198,370.528198,0.007687,0.007687,NaN,NaN,NaN,None
140826,25772622312114946,2.389303,-0.055498,25772622312114382,False,False,False,False,False,False,...,23.106299,23.106299,272.487525,272.487525,0.142426,0.142426,1658.255613,23.350871,NaN,None


In [9]:
all_sources_df.to_parquet("/sdf/home/n/ncaplar/github/notebooks_lf/PPDB_2026/single_visit_star_footprints_sample.parquet", index=False)

In [10]:
parquet_path = "/sdf/home/n/ncaplar/github/notebooks_lf/PPDB_2026/single_visit_star_footprints_sample.parquet"
pf = pq.ParquetFile(parquet_path)

rg = 0
obj_lc_parquet_file = pf  # <- use your file here

obj_lc_cols = obj_lc_parquet_file.metadata.num_columns
obj_lc_sizes = np.zeros(obj_lc_cols, dtype=np.int64)
row_groups = obj_lc_parquet_file.metadata.num_row_groups
obj_lc_lens = np.zeros(obj_lc_cols, dtype=np.int64)

for rg in range(row_groups):
    rg_meta = obj_lc_parquet_file.metadata.row_group(rg)
    for col in range(obj_lc_cols):
        col_meta = rg_meta.column(col)
        obj_lc_sizes[col] += col_meta.total_compressed_size
        obj_lc_lens[col] += col_meta.num_values

first_row_group = obj_lc_parquet_file.metadata.row_group(0)

column_names = [
    first_row_group.column(col).path_in_schema.removesuffix(".list.element")
    for col in range(first_row_group.num_columns)
]

column_types = [
    first_row_group.column(col).physical_type
    for col in range(first_row_group.num_columns)
]

obj_lc_frame = pd.DataFrame(
    {
        "name": column_names,
        "size": obj_lc_sizes,
        "length": obj_lc_lens,
        "type": column_types,
    }
)
obj_lc_frame["percent"] = obj_lc_frame["size"] / obj_lc_frame["size"].sum() * 100
obj_lc_frame["density"] = obj_lc_frame["size"] / obj_lc_frame["length"]
obj_lc_frame = obj_lc_frame.sort_values("density", ascending=False)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", 200)

obj_lc_frame

,name,size,length,type,percent,density
478,base_LocalBackground_fluxErr,1424355,140837,DOUBLE,0.517065,10.113500
476,base_LocalBackground_flux,1424355,140837,DOUBLE,0.517065,10.113500
479,base_LocalBackground_magErr,1424355,140837,DOUBLE,0.517065,10.113500
294,base_LocalBackground_instFlux,1424355,140837,DOUBLE,0.517065,10.113500
295,base_LocalBackground_instFluxErr,1424355,140837,DOUBLE,0.517065,10.113500
322,base_PsfFlux_instFlux,1424039,140837,DOUBLE,0.516951,10.111256
323,slot_PsfFlux_instFlux,1424039,140837,DOUBLE,0.516951,10.111256
488,base_PsfFlux_flux,1424039,140837,DOUBLE,0.516951,10.111256
495,slot_PsfFlux_magErr,1424039,140837,DOUBLE,0.516951,10.111256
494,base_PsfFlux_magErr,1424039,140837,DOUBLE,0.516951,10.111256


In [11]:
# 277 mb for 100 single_visit_star_footprints, which is about 2.77 mb per footprint on average.
# that is per visit, per detector 

# in GB per night 
2.77 * 187 * 800/1024

404.6796875

In [3]:
2+2

4

In [4]:
import os

os.environ["no_proxy"] += ",.consdb"

from lsst.summit.utils import ConsDbClient

client = ConsDbClient("http://consdb-pq.consdb:8080/consdb")

instrument = 'lsstcam'

visits_query = f'''
    SELECT * FROM cdb_{instrument}.visit1 WHERE day_obs >= 20250415 and science_program = 'BLOCK-365'
'''

visits = client.query(visits_query).to_pandas()

ConnectionError: HTTPConnectionPool(host='consdb-pq.consdb', port=8080): Max retries exceeded with url: /consdb/query (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x7f98563c12b0>: Failed to resolve 'consdb-pq.consdb' ([Errno -2] Name or service not known)"))

In [18]:

refs = list(
    butler.registry.queryDatasets(
        "single_visit_star_footprints",
        collections=collection,
        where=(
            "instrument='LSSTCam' "
            "AND day_obs = day "
            "AND physical_filter = band "
            "AND detector = det"
        ),
        bind={"day": 20260128, "band": "r_57", "det": 187},
    )
)

print(f"Found {len(refs)} refs")

Found 74 refs


In [19]:
refs


[DatasetRef(DatasetType('single_visit_star_footprints', {band, instrument, day_obs, detector, physical_filter, visit}, SourceCatalog), {instrument: 'LSSTCam', detector: 187, visit: 2026012800236, band: 'r', day_obs: 20260128, physical_filter: 'r_57'}, run='LSSTCam/runs/nightlyValidation/37', id=019c07f0-7f7e-7759-83cc-b6849814d1df),
 DatasetRef(DatasetType('single_visit_star_footprints', {band, instrument, day_obs, detector, physical_filter, visit}, SourceCatalog), {instrument: 'LSSTCam', detector: 187, visit: 2026012800237, band: 'r', day_obs: 20260128, physical_filter: 'r_57'}, run='LSSTCam/runs/nightlyValidation/37', id=019c07f5-ccbd-7888-810e-b6b88a99989a),
 DatasetRef(DatasetType('single_visit_star_footprints', {band, instrument, day_obs, detector, physical_filter, visit}, SourceCatalog), {instrument: 'LSSTCam', detector: 187, visit: 2026012800238, band: 'r', day_obs: 20260128, physical_filter: 'r_57'}, run='LSSTCam/runs/nightlyValidation/37', id=019c07f2-30a5-7fa7-a2d2-12245f95b2